In [ ]:
import os
import yfinance as  yf
import pandas as pd
import numpy as np
import cvxpy as cp
from scipy.stats import norm
import matplotlib.pyplot as plt
from hmmlearn.hmm import GaussianHMM

In [ ]:
class HMM:
  def __init__(self, hmm_params, debug, **kwargs):
    super().__init__(debug=debug, **kwargs)
    self.debug = debug
    self.hmm_params = hmm_params
    self.model = None

  def train(self, returns):
    if self.debug:
      print(returns.head(5))

    self.model = GaussianHMM(**self.hmm_params)
    self.model.fit(returns)
    self.state_i = self.model.predict(returns)[-1]
    self.state_probs = self.model.predict_proba(returns)[-1]

  def get_safe_tramsmat(self, transmat):
    transmat_safe = np.copy(transmat)
    for idx, row in enumerate(transmat_safe):
      row_sum = np.sum(row)
      if row_sum == 0 or np.isnan(row_sum):
        transmat_safe[idx] = np.full(len(row), 1.0 / len(row))
      else:
        transmat_safe[idx] = row / row_sum

    return transmat_safe

  def get_P_expected(self, recalc_win):
    if self.model is None:
        raise ValueError("HMM has not been fitted yet. Run train() first.")

    expected_probs = self.state_probs @ np.linalg.matrix_power(
                                        self.model.transmat_, recalc_win
                                  )
    return expected_probs

  def get_blend_cov(self):
    if self.model is None:
        raise ValueError("HMM has not been fitted yet. Run train() first.")

    raw_cov = self.model.covars_

    if raw_cov.ndim == 3:
        state_cov = raw_cov

    elif raw_cov.ndim == 2:
        n_comp, k_assets = raw_cov.shape
        state_cov = np.zeros((n_comp, k_assets, k_assets))

        for s in range(n_comp):
            state_cov[s] = np.diag(raw_cov[s])

    else:
        raise ValueError(f"Unexpected covars_ shape: {raw_cov.shape}")

    prob_next_states = self.model.transmat_[self.state_i]
    transmat_safe = self.get_safe_tramsmat(self.model.transmat_)

    n_comp, k_assets = self.model.means_.shape
    blend_er = prob_next_states @ self.model.means_

    blend_cov = np.zeros((k_assets, k_assets))

    for s in range(n_comp):
        shift = (self.model.means_[s] - blend_er).reshape(-1, 1)
        shift_unc = shift @ shift.T
        blend_cov += prob_next_states[s] * (state_cov[s] + shift_unc)

    return blend_cov




In [ ]:
class Filter:
  def __init__(
      self,
      rf:float=0.01,
      T:int=252,
      corr_threshold:int=.85,
      debug:bool=False,
      **kwargs
  ):
    super().__init__(debug=debug, **kwargs)
    self.corr_threshold = corr_threshold
    self.rf = rf/T
    self.debug = debug

  def hmm_filter(
      self,
      returns:pd.DataFrame,
      hmm:HMM,
      risk_q:float,
      recalc_window:int,
      selection_q:float=0.6
    ):
    self.hmm = hmm
    scores = self.score_assets(recalc_window, risk_q)
    score_series = pd.Series(scores, index=returns.columns)

    threshold = score_series.quantile(selection_q)
    valid_assets = score_series[score_series > threshold].index.tolist()

    return returns[valid_assets]

  def score_assets(self, recalc_win, risk_q):
    if self.hmm.model is None:
      raise ValueError("HMM has not been fitted yet. Run train() first.")

    P_expected = self.hmm.get_P_expected(recalc_win)
    scores = self._calcualate_score(risk_q, P_expected)

    return scores

  def _calcualate_score(self, risk_q, P_exp):
    if self.hmm.model.covars_.ndim == 2:
      asset_vars = self.hmm.model.covars_
    elif self.hmm.model.covars_.ndim == 3:
      asset_vars = np.diagonal(self.hmm.model.covars_, axis1=1, axis2=2)
    else:
      asset_vars = self.hmm.model.covars_

    tail_params = norm.pdf(norm.ppf(risk_q)) / (1 - risk_q)
    ES = -self.hmm.model.means_ + np.sqrt(asset_vars) * tail_params

    blend_es = P_exp @ ES
    blend_mean = P_exp @ self.hmm.model.means_

    tres = blend_mean / np.maximum(blend_es, 1e-6)
    return tres

  def corr_filter(self, returns:pd.DataFrame):
    corr_matrix = returns.corr()
    std = returns.std()

    sharpe = (np.mean(returns)-self.rf) / std

    drop = []
    for ticker in returns.columns:
      if ticker in drop:
        continue
      ticker_idx = returns.columns.get_loc(ticker)

      for i in returns.columns:
        if ticker == i or i in drop:
          continue

        i_idx = returns.columns.get_loc(i)
        if corr_matrix.iloc[ticker_idx, i_idx] > self.corr_threshold:
          if self.debug:
            print(corr_matrix.iloc[ticker_idx, i_idx])
          if sharpe[ticker] > sharpe[i]:
            drop.append(i)
          else:
            drop.append(ticker)

    return returns.drop(columns=drop)



In [ ]:
def optimize_portfolio(
    returns,
    sector_exposure_matrix,
    country_exposure_matrix,
    max_sector_exposure,
    max_country_exposure,
    per_sector_penalty=None,
    per_country_penalty=None,
    sector_penalty_weight=0.1,
    country_penalty_weight=0.05,
    prct=0.95,
    risk_aversion=3.0,
    min_weight_threshold=0.01
):
    if per_sector_penalty is None:
        per_sector_penalty = np.ones(sector_exposure_matrix.shape[0])
    else:
        per_sector_penalty = np.array(per_sector_penalty)

    if per_country_penalty is None:
        per_country_penalty = np.ones(country_exposure_matrix.shape[0])
    else:
        per_country_penalty = np.array(per_country_penalty)

    mu = np.mean(returns, axis=0)
    S, N = returns.shape
    R = returns.values

    w = cp.Variable(N)
    u = cp.Variable(S)
    zeta = cp.Variable()

    ex_R = mu.values @ w
    es = zeta + (1 / ((1 - prct) * S)) * cp.sum(u)

    constraints = [
        u >= -R @ w - zeta,
        u >= 0,
        cp.sum(w) == 1,
        w >= 0,
        sector_exposure_matrix @ w <= max_sector_exposure,
        country_exposure_matrix @ w <= max_country_exposure,
    ]

    sector_Q = np.diag(per_sector_penalty)
    sector_P = sector_exposure_matrix.T @ sector_Q @ sector_exposure_matrix

    sector_penalty_val = cp.quad_form(w, sector_P)

    country_Q = np.diag(per_country_penalty)
    country_P = country_exposure_matrix.T @ country_Q @ country_exposure_matrix
    country_penalty_val = cp.quad_form(w, country_P)

    total_penalties = (
        sector_penalty_weight * sector_penalty_val
        + country_penalty_weight * country_penalty_val
    )

    obj_fn = cp.Maximize(ex_R - risk_aversion * es - total_penalties)

    problem = cp.Problem(obj_fn, constraints)
    problem.solve(solver=cp.CLARABEL)


    if problem.status not in ["optimal", "feasible"]:
        raise ValueError(f"Optimization failed: {problem.status}")

    weights = np.array(w.value)
    weights[weights < min_weight_threshold] = 0.0

    total_w = np.sum(weights)
    if total_w > 0:
      weights_recalc = weights / total_w

    return weights_recalc

In [ ]:
class DataStore:
  def __init__(self, debug:bool=False, **kwargs):
    super().__init__(
        debug=debug,
        **kwargs
    )
    self.debug = debug

  def download(self, indices:str, start:str, end:str, interval:str):
    data = {}
    for ticker in indices:
        clean_ticker = ticker.strip()
        try:
            tmp_data = yf.Ticker(clean_ticker) \
                          .history(
                              start=start,
                              end=end,
                              interval=interval
                          )["Close"]

            if tmp_data.empty:
                print(f"Warning: No data returned for {clean_ticker}")
                continue
            tmp_data.index = pd.to_datetime(tmp_data.index)\
                                    .tz_localize(None)\
                                    .normalize()
            tmp_data = tmp_data[~tmp_data.index.duplicated(keep='last')]
            data[clean_ticker] = tmp_data
        except Exception as e:
            print(f"Failed to download {clean_ticker}: {e}")

    if not data:
        raise ValueError(
            "No data was successfully downloaded  \
              for any of the provided tickers."
        )
    df = pd.concat(data, axis=1)
    return df.sort_index()

  def get_data(
      self,
      indices:list,
      start:str,
      end:str,
      interval:str="1d",
      benchmark:str="^GSPC"
  ):
    self.benchmark_ticker = benchmark
    df_path = f"portfolio_{start}_{end}.parquet"
    benchmark_path = f"benchmark_{start}_{end}.parquet"

    if not os.path.exists(df_path) or not os.path.exists(benchmark_path):
        df = self.download(indices, start, end, interval)
        bench_data = self.download([benchmark], start, end, interval)

        df.to_parquet(df_path)
        bench_data.to_parquet(benchmark_path)

    else:
        df = pd.read_parquet(df_path)
        bench_data= pd.read_parquet(benchmark_path)

    self.returns_raw = df.pct_change().dropna()
    self.benchmark = bench_data.pct_change().dropna()

  def plot_data(self):
    (np.cumsum(self.returns_raw * 100, axis=0) + 100).plot(figsize=(15, 10))
    plt.show()

  def plot_benchmark(self):
    (np.cumsum(self.benchmark * 100, axis=0) + 100).plot(figsize=(15, 10))
    plt.show()

In [ ]:
class Optimizer:
  def __init__(
      self,
      debug:bool=False,
      **kwargs
    ):
    super().__init__(**kwargs)
    self.debug = debug

  def optimize(
      self,
      returns,
      regime:str,
      risk_aversion:float=None,
      sector_exposure_matrix=None,
      country_exposure_matrix=None,
      blended_mtx=None
    ):
    if (
          sector_exposure_matrix is None
          and country_exposure_matrix is None
          and blended_mtx is None
    ):
      raise ValueError()

    sec_exp_cap = np.array(list(self.max_sector_exposure.values()))
    cnt_exp_cap = np.array(list(self.max_country_exposure.values()))

    if sector_exposure_matrix is not None:
        sector_exposure_matrix = np.asarray(sector_exposure_matrix, dtype=float)
    if country_exposure_matrix is not None:
        country_exposure_matrix = np.asarray(
            country_exposure_matrix, dtype=float
        )

    if blended_mtx is not None:
        blended_mtx = np.asarray(blended_mtx, dtype=float)

    S, N = returns.shape
    R = returns.values

    w, u, zeta = cp.Variable(N), cp.Variable(S), cp.Variable()

    es = zeta + (1 / ((1 - self.prct) * S)) * cp.sum(u)

    constraints = [
        u >= -R @ w - zeta,
        u >= 0,
        cp.sum(w) == 1,
        w >= 0,
        sector_exposure_matrix @ w <= sec_exp_cap,
        country_exposure_matrix @ w <= cnt_exp_cap,
    ]

    if regime == "Low" or self.debug:
      w = self.mvo_optimizer(
        returns,
        es,
        constraints,
        w,
        risk_aversion,
        sector_exposure_matrix,
        country_exposure_matrix
      )

    else:
      w = self.gmv_optimizer(
          constraints=constraints,
          w=w,
          blended_cov_mtx=blended_mtx
      )

    weights = np.array(w.value)
    weights[weights < self.min_weight_threshold] = 0.0
    weights_recalc = weights / np.sum(weights)

    return weights_recalc


  def mvo_optimizer(
      self,
      returns,
      es,
      constraints,
      w,
      risk_aversion,
      sector_exposure_matrix,
      country_exposure_matrix
    ):
    if self.per_sector_penalty is None:
        per_sector_penalty_mtx = np.ones(sector_exposure_matrix.shape[0])
    else:
        per_sector_penalty_mtx = np.array(
            list(self.per_sector_penalty.values())
        )

    if self.per_country_penalty is None:
        per_country_penalty_mtx = np.ones(country_exposure_matrix.shape[0])
    else:
        per_country_penalty_mtx = np.array(
            list(self.per_country_penalty.values())
        )

    mu = np.mean(returns, axis=0).values
    ex_R = mu @ w

    sector_Q = np.diag(per_sector_penalty_mtx)
    sector_P = sector_exposure_matrix.T @ sector_Q @ sector_exposure_matrix
    sector_penalty_val = cp.quad_form(w, sector_P)

    country_Q = np.diag(per_country_penalty_mtx)
    country_P = country_exposure_matrix.T @ country_Q @ country_exposure_matrix
    country_penalty_val = cp.quad_form(w, country_P)

    total_penalties = (
        self.sector_penalty_weight * sector_penalty_val
        + self.country_penalty_weight * country_penalty_val
    )

    obj_fn = cp.Maximize(ex_R - risk_aversion*es - total_penalties)
    problem = cp.Problem(obj_fn, constraints)
    problem.solve(solver=cp.CLARABEL)

    if problem.status not in ["optimal", "feasible"]:
        raise ValueError(f"Optimization failed: {problem.status}")

    return w

  def gmv_optimizer(self, constraints, w, blended_cov_mtx):
    obj_fn = cp.Minimize(0.5 * w @ blended_cov_mtx @ w.T)
    problem = cp.Problem(obj_fn, constraints)
    problem.solve(solver=cp.CLARABEL)

    if problem.status not in ["optimal", "feasible"]:
        raise ValueError(f"Optimization failed: {problem.status}")

    return w

In [ ]:
class PortfolioGenerator(HMM, Filter,  DataStore, Optimizer):
    def __init__(
        self,
        hmm_params:dict,
        rf:float=0.01,
        T:int=252,
        recalc_window:int=21,
        risk_q:float=0.95,
        corr_threshold:int=.85,
        debug:bool=True
    ):
      super().__init__(
        hmm_params=hmm_params,
        debug=debug,
        rf=rf,
        T=T,
        corr_threshold=corr_threshold
      )
      self.sector_penalty_weight = 0.1
      self.country_penalty_weight = 0.05
      self.prct = 0.95
      self.risk_aversion = 3.0
      self.min_weight_threshold = 0.01
      self.debug = debug
      self.returns = None
      self.recalc_window = recalc_window
      self.risk_q = risk_q

      self.filterred = False

      self.max_sector_exposure = {
          "Tech": 0.30, "Finance": 0.20, "Industry": 0.20,
          "Telecom": 0.10, "ConsumerGoods": 0.30, "Health": 0.2
      }
      self.max_country_exposure = {"US": 0.7, "Europe": 0.25, "Asia": 0.25}
      self.per_sector_penalty = {
          "Tech": 1.3, "Finance": 1.8, "Industry": 1.0,
          "Telecom": 0.8, "ConsumerGoods": 0.8, "Health": 0.3
      }
      self.per_country_penalty = {"US": 1.5, "Europe": 1.0, "Asia": 1.0}

    def train_hmm(self):
      super().train(self.returns or self.returns_raw)

    def hmm_filter(self):
      if self.filterred:
        raise ValueError("Filter has already been applied")

      if self.model is None:
        raise ValueError("HMM has not been fitted yet. Run train() first.")

      self.filterred = True
      self.returns = super().hmm_filter(
          returns=self.returns_raw,
          recalc_window=self.recalc_window,
          risk_q=self.risk_q,
          hmm=self
      )

      del self.returns_raw

    def corr_filter(self, ):
      if self.filterred:
        raise ValueError("Filter has already been applied")

      self.filterred = True

      self.returns =  super().corr_filter(self.returns_raw)
      del self.returns_raw

    def get_current_regime(self):
      P_expected = self.get_P_expected(self.recalc_window)

      covars = self.model.covars_

      if covars.ndim == 3:
        state_variances = np.mean(np.diagonal(covars, axis1=1, axis2=2), axis=1)
      else:
        state_variances = np.mean(covars, axis=1)

      sorted_state_indices = np.argsort(state_variances)

      n_states = len(sorted_state_indices)

      if n_states == 2:
        regime_mapping = {
          sorted_state_indices[0].item(): "Low",
          sorted_state_indices[1].item(): "High"
        }
      elif n_states == 3:
        regime_mapping = {
          sorted_state_indices[0].item(): "Low",
          sorted_state_indices[1].item(): "Neutral",
          sorted_state_indices[2].item(): "High"
        }
      else:
          regime_mapping = {idx.item(): f"State_{idx.item()}" for idx in sorted_state_indices}

      dominant_raw_idx = int(np.argmax(P_expected))

      return regime_mapping[dominant_raw_idx]

    def get_exposure(self, exposures, country_alloc):
      exposure_df = pd.DataFrame.from_dict(exposures, orient='index')
      exposure_df.columns = [
          "Tech", "Finance", "Industry", "Telecom", "ConsumerGoods", "Health"
      ]
      country_df = pd.DataFrame.from_dict(country_alloc, orient='index')
      country_df.columns = ["US", "Europe", "Asia"]
      return exposure_df, country_df

    def optimize_portfolio(self, exposures, country_alloc):
      regime = self.get_current_regime()
      blend_mtx = None if regime == "Low" \
                  else self.get_blend_cov(self.recalc_window)

      sector_matrix, country_matrix = self.get_exposure(
          exposures, country_alloc
      )

      sector_mtx_vals = sector_matrix.loc[self.returns.columns].T.values
      country_mtx_vals = country_matrix.loc[self.returns.columns].T.values

      w = self.optimize(
        returns=self.returns,
        regime=regime,
        risk_aversion=self.risk_aversion,
        sector_exposure_matrix=sector_mtx_vals,
        country_exposure_matrix=country_mtx_vals,
        blended_mtx=blend_mtx
      )

      return w




In [ ]:
class PortfolioBacktest(PortfolioGenerator):
  def __init__(self, 
        hmm_params:dict,
        rf:float=0.01,
        T:int=252,
        recalc_window:int=21,
        risk_q:float=0.95,
        corr_threshold:int=.85,
        debug=True
    ):
    super().init(
        hmm_params=hmm_params,
        rf=rf,
        T=T,
        recalc_window=recalc_window,
        risk_q=risk_q,
        corr_threshold=corr_threshold,
        debug=debug
    )

  def calculate_metrics(self, returns_series, trading_days=252):
    cum_returns = (1 + returns_series).cumprod()
    final_return = cum_returns.iloc[-1] - 1
    ann_return = returns_series.mean() * trading_days
    ann_vol = returns_series.std() * np.sqrt(trading_days)
    sharpe = ann_return / ann_vol if ann_vol != 0 else 0

    running_max = cum_returns.cummax()
    drawdowns = (cum_returns - running_max) / running_max
    max_dd = drawdowns.min()

    return {
        "Final Return": final_return,
        "Sharpe Ratio": sharpe,
        "Max Drawdown": max_dd
    }

  def apply_lot_size_rounding(self, weights, asset_prices, total_portfolio_value=1000):
    prices = np.array(asset_prices)
    ideal_cash_alloc = weights * total_portfolio_value
    ideal_shares = ideal_cash_alloc / prices
    actual_shares = np.round(ideal_shares)
    actual_cash_alloc = actual_shares * prices
    actual_total_spent = np.sum(actual_cash_alloc)

    rounded_weights = actual_cash_alloc / actual_total_spent if actual_total_spent > 0 else weights
    return rounded_weights, actual_shares

  def rolling_backtest(self, sector_exp, country_exp, train_window=252, recalc_window=21):
    total_periods = len(self.returns)
    oos_portfolio_returns = []
    oos_benchmark_returns = []
    oos_dates = []

    sector_matrix, country_matrix = self.get_exposure(sector_exp, country_exp)

    hmm_engine = HMM(self.hmm_params)

    for start_idx in range(0, total_periods - train_window, recalc_window):
    train_end_idx = start_idx + train_window
    test_end_idx = min(train_end_idx + recalc_window, total_periods)

    train_returns_raw = self.returns.iloc[start_idx:train_end_idx]

    train_returns = Selector().corr_filter(train_returns_raw)

    test_returns = self.returns.loc[:, train_returns.columns].iloc[train_end_idx:test_end_idx]
    if test_returns.empty:
      break

    test_bench = self.benchmark.reindex(test_returns.index).fillna(0)

    sector_mtx_vals = sector_matrix.loc[train_returns.columns].T.values
    country_mtx_vals = country_matrix.loc[train_returns.columns].T.values

    _, expected_probs = hmm_engine.hmm_score(
      returns=train_returns,
      hmm_params=self.hmm_params,
      risk_q=self.prct,
      recalc_window=recalc_window
    )

    predicted_regime = self.get_current_regime(hmm_engine.hmm, expected_probs)

    if self.debug:
      print(f"Date: {train_returns.index[-1].strftime('%Y-%m-%d')} | Fwd Prob Vector: {np.round(expected_probs, 3)} -> Strategy: {predicted_regime}")

    if predicted_regime == "High":
      blended_cov_mtx = hmm_engine.get_blend_cov()
    else:
      blended_cov_mtx = None

    optimal_w = self.optimize_portfolio(
      returns=train_returns,
      sector_exposure_matrix=sector_mtx_vals,
      country_exposure_matrix=country_mtx_vals,
      blended_mtx=blended_cov_mtx,
      regime=predicted_regime,
      debug=self.debug
    )

    optimal_w_scaled, _ = self.apply_lot_size_rounding(optimal_w, test_returns.iloc[-1])
    period_oos_returns = test_returns.values @ optimal_w_scaled

    oos_portfolio_returns.extend(period_oos_returns)
    oos_benchmark_returns.extend(test_bench.iloc[:, 0].values)
    oos_dates.extend(test_returns.index)

    strat_perf = pd.Series(oos_portfolio_returns, index=oos_dates)
    bench_perf = pd.Series(oos_benchmark_returns, index=oos_dates)

    strat_metrics = self.calculate_metrics(strat_perf)
    bench_metrics = self.calculate_metrics(bench_perf)

    summary_df = pd.DataFrame(
      {
      "Strategy": [
          f"{strat_metrics['Final Return']*100:.2f}%",
          f"{strat_metrics['Sharpe Ratio']:.2f}",
          f"{strat_metrics['Max Drawdown']*100:.2f}%",
      ],
      f"{self.benchmark_ticker} Benchmark": [
          f"{bench_metrics['Final Return']*100:.2f}%",
          f"{bench_metrics['Sharpe Ratio']:.2f}",
          f"{bench_metrics['Max Drawdown']*100:.2f}%",
      ],
      },
      index=["Final Return", "Risk-Adjusted Return (Sharpe)", "Worst Drawdown"],
    )

    self.display_results(summary_df, strat_perf, bench_perf)
    return optimal_w, train_returns.columns

  def display_results(self, summary_table, strat_series, bench_series):
    print(summary_table)
    plt.figure(figsize=(12, 6))
    plt.plot(np.cumsum(strat_series) * 100 + 100, label="Strategy (Regime-Switched)")
    plt.plot(np.cumsum(bench_series) * 100 + 100, label="Benchmark")
    plt.legend()
    plt.show()

In [ ]:
returns = data.pct_change().dropna()
benchmark = sp500.pct_change().dropna()

summary_table, strat_series, bench_series, w, cols = rolling_backtest(
    returns=returns,
    benchmark=benchmark,
    sector_matrix=exposure_df,
    country_matrix=country_df,
    per_sector_penalty=per_sector_penalty,
    per_country_penalty=per_country_penalty,
    max_sector=max_sector_exposure,
    max_country=max_country_exposure,
    train_window=252,
    recalc_window=21
)


print(summary_table)
plt.plot(np.cumsum(strat_series)*100+100, label="strat")
plt.plot(np.cumsum(bench_series)*100+100, label="bench")
plt.legend()